# 02. 건강 분야 문단 모음 + 임베딩 + FAISS (RAG 과제)

MS MARCO QnA v2.1에서 **건강 분야 질문**만 골라 검색 대상 문단 모음과 평가 세트를 만들고, 임베딩·FAISS 인덱스를 만든 뒤 검색 성능을 잽니다.

| 파일 | 내용 |
|---|---|
| `corpus.jsonl` | 검색 대상 문단 약 46만 개 `{"pid", "text", "url", "src"}` |
| `eval.jsonl` | 평가 질문 650개 = 건강·답 있음 500 + 건강·답 없음 100 + 분야 밖 50 |
| `train_health_qa.jsonl.gz` | train의 건강 질문 전체 (다음 단계 파인튜닝용) |
| `corpus_emb_bge.npy` | 문단 임베딩 (float16) |
| `faiss_bge_flatip.index` | FAISS 인덱스 |
| `retrieval_results.json` | BM25 / 임베딩 검색 점수 |

- **런타임: A100 + 고용량 RAM**. 위에서부터 순서대로 실행하세요.
- 데이터 파일을 직접 받을 필요는 없습니다. Hugging Face에서 자동으로 받습니다(약 2분).
- `01_preprocess.ipynb`를 먼저 돌리지 않아도 됩니다. 이 노트북이 전처리부터 다시 합니다.

## 1. 설치

In [2]:
!pip -q install sentence-transformers faiss-cpu bm25s

## 2. 저장 위치

드라이브 연결이 되면 드라이브에, 안 되면 Colab 서버 안(`/content`)에 저장합니다.
**서버 안에 저장한 파일은 서버가 꺼지면 사라집니다.**

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/rag_msmarco/health'
except Exception as e:
    print('드라이브 연결 실패 → 서버 안에 저장합니다:', e)
    SAVE_DIR = '/content/rag_msmarco/health'
print('저장 위치:', SAVE_DIR)

Mounted at /content/drive
저장 위치: /content/drive/MyDrive/rag_msmarco/health


## 3. 설정

In [4]:
import os, re, json, gzip, random, time
from collections import Counter
from urllib.parse import urlparse
import numpy as np

SEED             = 42
MIN_HEALTH_HITS  = 2       # 질문에 딸린 문단 10개 중 건강 사이트 문단이 이 개수 이상이면 건강 질문
N_TRAIN_HEALTH_Q = 25000   # 문단을 가져올 train 건강 질문 수 (약 25만 문단)
N_DISTRACTOR_Q   = 10000   # 방해 문단을 가져올 dev 일반 질문 수 (약 10만 문단)
N_EVAL_ANS       = 500     # 평가: 건강·답 있음
N_EVAL_NOANS     = 100     # 평가: 건강·답 없음 ("모르면 모른다"고 하는지)
N_EVAL_OUT       = 50      # 평가: 분야 밖 질문 (건강 질문이 아니면 거절하는지)
NO_ANSWER        = 'No Answer Present.'

EMB_MODEL    = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '   # bge 질문용 접두어

os.makedirs(SAVE_DIR, exist_ok=True)

def save_jsonl(rows, path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'wt', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

### 건강 사이트 목록

dev 세트 문단 100만 개의 출처를 세어 보고, 많이 나오는 의료·건강 사이트를 골랐습니다.

In [5]:
HEALTH_SITES = [
    'webmd.com', 'mayoclinic.org', 'medicinenet.com', 'medical-dictionary.thefreedictionary.com',
    'verywell.com', 'healthline.com', 'drugs.com', 'livestrong.com', 'healthtap.com', 'cdc.gov',
    'rightdiagnosis.com', 'emedicinehealth.com', 'medscape.com', 'everydayhealth.com',
    'medlineplus.gov', 'nhs.uk', 'rxlist.com', 'sharecare.com', 'medicalnewstoday.com',
    'kidshealth.org', 'nih.gov', 'patient.info', 'labtestsonline.org', 'aafp.org',
    'clevelandclinic.org', 'hopkinsmedicine.org', 'healthgrades.com', 'familydoctor.org',
    'merckmanuals.com', 'uptodate.com', 'health.harvard.edu', 'babycenter.com', 'psychologytoday.com',
]

def host(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h

def is_health(url):
    h = host(url)
    return any(h == s or h.endswith('.' + s) for s in HEALTH_SITES)

print(is_health('http://www.webmd.com/a'), is_health('https://my.clevelandclinic.org/b'), is_health('https://en.wikipedia.org/c'))

True True False


## 4. 데이터 받기

In [6]:
from datasets import load_dataset

ds = load_dataset('microsoft/ms_marco', 'v2.1')
dev, train = ds['validation'], ds['train']
print(ds)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/9.48k [00:00<?, ?B/s]

v2.1/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  210MB            

v2.1/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

v2.1/train-00000-of-00007.parquet: reconstructing file:   0%|          |  0.00B /  240MB            

...

v2.1/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/101093 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/808731 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/101092 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 101093
    })
    train: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 808731
    })
    test: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 101092
    })
})


## 5. 정리 함수 + 건강 질문 고르기

- 질문 앞의 `". "` 같은 쓰레기 문자 제거, 문단의 줄바꿈·연속 공백 정리 (01 노트북과 같음)
- 질문마다 건강 사이트 문단 수(`hits`)를 셉니다.

In [7]:
def clean_query(q):
    q = ' '.join(q.split())
    return re.sub(r'^[\s.,;:!?\-]+', '', q)

def clean_text(t):
    return ' '.join(t.split())

def clean_list(v):
    return [clean_text(a) for a in v if a and a.strip()]

def count_hits(split):
    hits = []
    for b in split.select_columns(['passages']).iter(batch_size=5000):
        hits += [sum(is_health(u) for u in p['url']) for p in b['passages']]
    return np.array(hits)

def iter_rows(split, idx):
    """idx 순서대로 한 행씩 dict로 돌려줍니다."""
    for b in split.select(list(map(int, idx))).iter(batch_size=1000):
        for j in range(len(b['query'])):
            yield {k: b[k][j] for k in b}

t = time.time()
dev_hits, train_hits = count_hits(dev), count_hits(train)
print(f'{time.time() - t:.0f}초')
print('dev   건강 질문:', (dev_hits >= MIN_HEALTH_HITS).sum(), '/', len(dev))
print('train 건강 질문:', (train_hits >= MIN_HEALTH_HITS).sum(), '/', len(train))

109초
dev   건강 질문: 13494 / 101093
train 건강 질문: 119659 / 808731


## 6. 문단 모음 만들기

세 곳에서 문단을 모읍니다. 내용이 같은 문단은 하나만 남기고, pid는 0부터 붙입니다.

| `src` | 출처 | 역할 |
|---|---|---|
| `dev_health` | dev 건강 질문 전체의 문단 | 평가 질문의 정답 문단이 여기 있음 |
| `train_health` | train 건강 질문 일부의 문단 | 건강 문단을 늘려 검색을 어렵게 |
| `dev_other` | dev 일반 질문 일부의 문단 | 방해 문단 (분야 밖 문서) |

건강 질문에 딸린 문단은 출처가 건강 사이트가 아니어도 함께 넣습니다. 실제 검색 결과처럼 섞여 있어야 현실적입니다.

In [8]:
rng = random.Random(SEED)

dev_health_idx   = np.where(dev_hits >= MIN_HEALTH_HITS)[0]
dev_other_idx    = np.where(dev_hits == 0)[0].tolist()
train_health_idx = np.where(train_hits >= MIN_HEALTH_HITS)[0].tolist()

rng.shuffle(dev_other_idx)
distractor_idx = sorted(dev_other_idx[:N_DISTRACTOR_Q])
out_pool_idx   = dev_other_idx[N_DISTRACTOR_Q:]           # 분야 밖 평가 질문 후보 (문단을 넣지 않은 질문)
train_pick     = set(rng.sample(train_health_idx, N_TRAIN_HEALTH_Q))

corpus, pid_of, n_dup = [], {}, 0

def add_passages(p, src):
    """문단을 모음에 넣고, 정답 문단(is_selected=1)의 pid 목록을 돌려줍니다."""
    global n_dup
    gold = []
    for text, sel, url in zip(p['passage_text'], p['is_selected'], p['url']):
        text = clean_text(text)
        if not text:
            continue
        if text in pid_of:
            n_dup += 1
        else:
            pid_of[text] = len(corpus)
            corpus.append({'pid': len(corpus), 'text': text, 'url': url, 'src': src})
        if sel == 1 and pid_of[text] not in gold:
            gold.append(pid_of[text])
    return gold

def to_question(row, gold, domain):
    answers = clean_list(row['answers'])
    has_answer = bool(answers) and answers != [NO_ANSWER]
    return {
        'qid': int(row['query_id']),
        'query': clean_query(row['query']),
        'gold_pids': gold,
        'answers': answers,
        'well_formed': clean_list(row['wellFormedAnswers']),
        'answerable': has_answer and len(gold) > 0,
        'query_type': row['query_type'],
        'domain': domain,
    }

t = time.time()
dev_health_q = [to_question(r, add_passages(r['passages'], 'dev_health'), 'health')
                for r in iter_rows(dev, dev_health_idx)]

train_health_qa = []   # 파인튜닝용: train 건강 질문 전체 (문단 원문 포함)
for i, r in zip(train_health_idx, iter_rows(train, train_health_idx)):
    in_corpus = i in train_pick
    if in_corpus:
        add_passages(r['passages'], 'train_health')
    train_health_qa.append({
        'qid': int(r['query_id']), 'query': clean_query(r['query']),
        'answers': clean_list(r['answers']), 'well_formed': clean_list(r['wellFormedAnswers']),
        'query_type': r['query_type'], 'in_corpus': in_corpus,
        'passages': [{'text': clean_text(t), 'is_selected': s}
                     for t, s in zip(r['passages']['passage_text'], r['passages']['is_selected'])],
    })

for r in iter_rows(dev, distractor_idx):
    add_passages(r['passages'], 'dev_other')

print(f'{time.time() - t:.0f}초')
print(f'문단 {len(corpus):,}개 (중복 {n_dup:,}개 제거)')
print('출처별:', Counter(c['src'] for c in corpus))

24초
문단 459,067개 (중복 25,666개 제거)
출처별: Counter({'train_health': 229697, 'dev_health': 131143, 'dev_other': 98227})


## 7. 평가 세트 고르기

- 건강·답 있음 500: 정답 문장이 있고 정답 문단도 1개 이상
- 건강·답 없음 100: `answers == ["No Answer Present."]`이고 정답 문단 0개
- 분야 밖 50: 건강 사이트 문단이 하나도 없는 dev 질문. 이 질문들의 문단은 모음에 넣지 않았습니다.

In [9]:
ans_pool   = [q for q in dev_health_q if q['answerable']]
noans_pool = [q for q in dev_health_q if q['answers'] == [NO_ANSWER] and not q['gold_pids']]
print('후보 수 - 답 있음:', len(ans_pool), ' 답 없음:', len(noans_pool))

rng = random.Random(SEED)
out_q = [to_question(r, [], 'out') for r in iter_rows(dev, rng.sample(out_pool_idx, N_EVAL_OUT))]
for q in out_q:
    q['answerable'] = False          # 분야 밖 → 거절해야 정답

eval_set = rng.sample(ans_pool, N_EVAL_ANS) + rng.sample(noans_pool, N_EVAL_NOANS) + out_q
print('평가 질문:', len(eval_set), Counter(q['domain'] for q in eval_set))
print('질문 종류:', Counter(q['query_type'] for q in eval_set))

후보 수 - 답 있음: 7996  답 없음: 5495
평가 질문: 650 Counter({'health': 600, 'out': 50})
질문 종류: Counter({'DESCRIPTION': 403, 'NUMERIC': 121, 'ENTITY': 105, 'PERSON': 12, 'LOCATION': 9})


## 8. 저장 + 확인

In [10]:
save_jsonl(corpus,          f'{SAVE_DIR}/corpus.jsonl')
save_jsonl(eval_set,        f'{SAVE_DIR}/eval.jsonl')
save_jsonl(train_health_qa, f'{SAVE_DIR}/train_health_qa.jsonl.gz')
with open(f'{SAVE_DIR}/meta.json', 'w') as f:
    json.dump({'source': 'huggingface microsoft/ms_marco v2.1', 'seed': SEED,
               'health_sites': HEALTH_SITES, 'min_health_hits': MIN_HEALTH_HITS,
               'n_passages': len(corpus), 'src_counts': Counter(c['src'] for c in corpus),
               'n_dev_health_q': len(dev_health_q), 'n_train_health_q': len(train_health_qa),
               'n_eval': {'answerable': N_EVAL_ANS, 'no_answer': N_EVAL_NOANS, 'out_of_domain': N_EVAL_OUT}},
              f, indent=2)

for name in ['corpus.jsonl', 'eval.jsonl', 'train_health_qa.jsonl.gz', 'meta.json']:
    print(name, f'{os.path.getsize(f"{SAVE_DIR}/{name}") / 1e6:.1f} MB')

# 확인: pid = 줄 번호, 정답 pid가 범위 안, 평가 질문 qid가 train 파인튜닝 데이터와 겹치지 않음
assert all(c['pid'] == i for i, c in enumerate(corpus))
assert all(0 <= p < len(corpus) for q in eval_set for p in q['gold_pids'])
assert not {q['qid'] for q in eval_set} & {q['qid'] for q in train_health_qa}, '평가 질문이 train에 있음'
print('검사 통과')

for q in random.Random(0).sample(ans_pool, 2):
    print('\n질문:', q['query'])
    print('정답:', q['answers'][0][:200])
    print('정답 문단:', corpus[q['gold_pids'][0]]['text'][:300])

corpus.jsonl 211.4 MB
eval.jsonl 0.2 MB
train_health_qa.jsonl.gz 137.7 MB
meta.json 0.0 MB
검사 통과

질문: what kind of bacteria is in a kindey infection
정답: E.coli bacteria
정답 문단: Causes of kidney infection. A kidney infection happens when bacteria infects your kidneys. The bacteria are usually a type called E. coli, which live in your bowel. The bacteria get in through the opening of the urethra and move upwards through your urinary tract, first infecting your bladder and th

질문: what is hydroxyzine pamoate
정답: Medication
정답 문단: How to use hydroxyzine pamoate. Take this medication by mouth with or without food as directed by your doctor, usually three or four times daily. The dosage is based on your age, medical condition, and response to treatment. In children, the dosage may also be based on weight.


## 9. 임베딩

- 모델: `BAAI/bge-base-en-v1.5` (768차원). 질문에만 접두어를 붙이고 문단은 그대로 넣습니다.
- 벡터를 정규화하므로 내적이 곧 코사인 유사도입니다.
- 이미 저장된 임베딩이 있으면 다시 계산하지 않고 불러옵니다.

**런타임이 끊긴 뒤 다시 왔다면** 1~3번 셀과 아래 셀부터 실행하면 됩니다. 저장된 파일을 불러옵니다.

In [11]:
if 'corpus' not in globals():
    corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
    eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
print('문단', len(corpus), '/ 평가 질문', len(eval_set))

문단 459067 / 평가 질문 650


In [12]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMB_MODEL, device='cuda' if torch.cuda.is_available() else 'cpu')
if model.device.type == 'cuda':
    model.half()
model.max_seq_length = 256
print(model.device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

EMB_PATH = f'{SAVE_DIR}/corpus_emb_bge.npy'
if os.path.exists(EMB_PATH) and len(np.load(EMB_PATH, mmap_mode='r')) == len(corpus):
    emb = np.load(EMB_PATH)
    print('저장된 임베딩 불러옴')
else:
    t = time.time()
    emb = model.encode([c['text'] for c in corpus], batch_size=512, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype('float16')
    np.save(EMB_PATH, emb)
    print(f'임베딩 {time.time() - t:.0f}초')
print(emb.shape, emb.dtype)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

cuda:0 NVIDIA A100-SXM4-40GB


Batches:   0%|          | 0/897 [00:00<?, ?it/s]

임베딩 166초
(459067, 768) float16


## 10. FAISS 인덱스

문단 50만 개 정도는 근사 검색 없이 전부 비교하는 `IndexFlatIP`로도 충분히 빠르고, 결과가 정확합니다.

In [13]:
import faiss

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb.astype('float32'))
faiss.write_index(index, f'{SAVE_DIR}/faiss_bge_flatip.index')
print('인덱스 문단 수:', index.ntotal)

인덱스 문단 수: 459067


## 11. 검색 평가 (BM25 vs 임베딩)

건강·답 있음 질문 500개로 잽니다.

- **Recall@k**: 상위 k개 안에 정답 문단이 하나라도 있는 질문의 비율
- **MRR@10**: 첫 정답 문단 순위의 역수 평균 (MS MARCO 공식 지표)

In [14]:
def retrieval_metrics(ranked, qs, ks=(1, 5, 10, 100)):
    res = {f'Recall@{k}': 0.0 for k in ks}
    res['MRR@10'] = 0.0
    for pids, q in zip(ranked, qs):
        gold = set(q['gold_pids'])
        for k in ks:
            res[f'Recall@{k}'] += bool(gold & set(pids[:k]))
        for r, p in enumerate(pids[:10]):
            if p in gold:
                res['MRR@10'] += 1 / (r + 1)
                break
    return {k: round(v / len(qs), 4) for k, v in res.items()}

eval_ans = [q for q in eval_set if q['answerable']]
queries  = [q['query'] for q in eval_ans]

In [15]:
# 임베딩 검색
q_emb = model.encode([QUERY_PREFIX + q for q in queries], batch_size=128,
                     normalize_embeddings=True, convert_to_numpy=True)
_, dense_ids = index.search(q_emb.astype('float32'), 100)
dense_scores = retrieval_metrics(dense_ids.tolist(), eval_ans)
print('임베딩(bge):', dense_scores)

임베딩(bge): {'Recall@1': 0.272, 'Recall@5': 0.592, 'Recall@10': 0.688, 'Recall@100': 0.938, 'MRR@10': 0.4007}


In [16]:
# BM25 검색 (단어 일치 기준선)
import bm25s

t = time.time()
bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize([c['text'] for c in corpus], stopwords='en'))
bm25_ids, _ = bm25.retrieve(bm25s.tokenize(queries, stopwords='en'), k=100)
bm25_scores = retrieval_metrics(bm25_ids.tolist(), eval_ans)
print(f'BM25 ({time.time() - t:.0f}초):', bm25_scores)

Split strings:   0%|          | 0/459067 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/459067 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/459067 [00:00<?, ?it/s]

Split strings:   0%|          | 0/500 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/500 [00:00<?, ?it/s]

BM25 (36초): {'Recall@1': 0.12, 'Recall@5': 0.316, 'Recall@10': 0.406, 'Recall@100': 0.71, 'MRR@10': 0.2016}


In [17]:
# 결과 표 + 질문 종류별 Recall@10 + 저장
import pandas as pd

table = pd.DataFrame({'BM25': bm25_scores, 'bge-base': dense_scores}).T
display(table)

by_type = {}
for name, ids in [('BM25', bm25_ids.tolist()), ('bge-base', dense_ids.tolist())]:
    for t_ in sorted({q['query_type'] for q in eval_ans}):
        sel = [(p, q) for p, q in zip(ids, eval_ans) if q['query_type'] == t_]
        by_type.setdefault(t_, {'n': len(sel)})[name] = retrieval_metrics([p for p, _ in sel], [q for _, q in sel])['Recall@10']
display(pd.DataFrame(by_type).T)

with open(f'{SAVE_DIR}/retrieval_results.json', 'w') as f:
    json.dump({'n_queries': len(eval_ans), 'n_passages': len(corpus),
               'bm25': bm25_scores, 'bge-base': dense_scores, 'recall@10_by_type': by_type}, f, indent=2)

,Recall@1,Recall@5,Recall@10,Recall@100,MRR@10
BM25,0.120,0.316,0.406,0.710,0.2016
bge-base,0.272,0.592,0.688,0.938,0.4007


,n,BM25,bge-base
DESCRIPTION,314.0,0.4172,0.7070
ENTITY,87.0,0.3793,0.6437
LOCATION,7.0,0.4286,0.5714
NUMERIC,86.0,0.3953,0.6628
PERSON,6.0,0.3333,0.8333


In [18]:
# 예시: 질문 하나의 검색 결과 상위 3개 (✅ = 정답 문단)
q = eval_ans[0]
print('질문:', q['query'], '\n정답:', q['answers'][0][:150], '\n')
for name, ids in [('BM25', bm25_ids[0]), ('bge-base', dense_ids[0])]:
    print(f'[{name}]')
    for p in ids[:3]:
        mark = '✅' if p in q['gold_pids'] else '  '
        print(f' {mark} [{p}] {corpus[p]["text"][:160]}')

질문: what causes boils to suddenly appear 
정답: Poor hygiene can cause armpit boils. 

[BM25]
    [153144] Boils Article. 1 Boils facts. 2 What is a boil? 3 What causes boils to form? 4 What are risk factors for boils? 5 How are boils diagnosed? 6 What are boil treat
    [6414] Recurrent boils appear suddenly? Please respond....worried. Discussion in 'Fibromyalgia Main Forum' started by JadeTortoise, Jan 30, 2005. JadeTortoise New Memb
    [155412] Symptoms of bacterial meningitis usually appear suddenly. Symptoms of viral meningitis may appear suddenly or develop gradually over a period of days. For examp
[bge-base]
    [6415] Pus head appears on the boil; Where do Boils Appear? Boils can occur anywhere on the skin, although they tend to develop on the face, neck, armpits, buttocks, o
    [155228] Boils can occur anywhere on the skin, although they tend to develop on the face, neck, armpits, buttocks, or thighs. They also occur most frequently in areas co
    [78204] There are many caus

In [19]:
# 결과 요약: 저장 파일 확인 + 점수 + results_summary.md 저장
EXPECTED = ['corpus.jsonl', 'eval.jsonl', 'train_health_qa.jsonl.gz', 'meta.json',
            'corpus_emb_bge.npy', 'faiss_bge_flatip.index', 'retrieval_results.json']
missing = [n for n in EXPECTED if not os.path.exists(f'{SAVE_DIR}/{n}')]
for n in EXPECTED:
    if n not in missing:
        print(f'  ✅ {n:28s} {os.path.getsize(f"{SAVE_DIR}/{n}") / 1e6:9.1f} MB')
print('빠진 파일:', missing or '없음')

def to_md(df):
    try:
        return df.to_markdown()
    except ImportError:
        return '```\n' + df.to_string() + '\n```'

summary = '\n'.join([
    f"# 검색 결과 요약 ({time.strftime('%Y-%m-%d %H:%M')})", '',
    f"- 문단 모음: {len(corpus):,}개 {dict(Counter(c['src'] for c in corpus))}",
    f"- 평가: 건강·답 있음 {len(eval_ans)}개로 검색 채점 (전체 평가 질문 {len(eval_set)}개)", '',
    to_md(table), '', '### 질문 종류별 Recall@10', '', to_md(pd.DataFrame(by_type).T),
])
with open(f'{SAVE_DIR}/results_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)
print('\n' + summary)

  ✅ corpus.jsonl                     211.4 MB
  ✅ eval.jsonl                         0.2 MB
  ✅ train_health_qa.jsonl.gz         137.7 MB
  ✅ meta.json                          0.0 MB
  ✅ corpus_emb_bge.npy               705.1 MB
  ✅ faiss_bge_flatip.index          1410.3 MB
  ✅ retrieval_results.json             0.0 MB
빠진 파일: 없음

# 검색 결과 요약 (2026-10-29 01:58)

- 문단 모음: 459,067개 {'dev_health': 131143, 'train_health': 229697, 'dev_other': 98227}
- 평가: 건강·답 있음 500개로 검색 채점 (전체 평가 질문 650개)

|          |   Recall@1 |   Recall@5 |   Recall@10 |   Recall@100 |   MRR@10 |
|:---------|-----------:|-----------:|------------:|-------------:|---------:|
| BM25     |      0.12  |      0.316 |       0.406 |        0.71  |   0.2016 |
| bge-base |      0.272 |      0.592 |       0.688 |        0.938 |   0.4007 |

### 질문 종류별 Recall@10

|             |   n |   BM25 |   bge-base |
|:------------|----:|-------:|-----------:|
| DESCRIPTION | 314 | 0.4172 |     0.707  |
| ENTITY      |  87 | 0.3793 |     0.

In [20]:
print(SAVE_DIR)

드라이브 저장 완료
